In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import  ColumnTransformer, make_column_transformer, make_column_selector
from sklearn.metrics import r2_score, log_loss, accuracy_score, roc_auc_score, f1_score, log_loss

from sklearn.tree import DecisionTreeRegressor, plot_tree, DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression, ElasticNet, Ridge
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from xgboost import XGBClassifier, XGBRegressor

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, StackingClassifier, StackingRegressor

from tqdm import tqdm  # Provides the progress of model running
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
concrete = pd.read_csv('D:/Machine_Learning/Cases/Concrete_Strength/Concrete_Data.csv')
concrete

,Cement,Blast,Fly,Water,Superplasticizer,Coarse,Fine,Age,Strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30
...,...,...,...,...,...,...,...,...,...
1025,276.4,116.0,90.3,179.6,8.9,870.1,768.3,28,44.28
1026,322.2,0.0,115.6,196.0,10.4,817.9,813.4,28,31.18
1027,148.5,139.4,108.6,192.7,6.1,892.4,780.0,28,23.70
1028,159.1,186.7,0.0,175.6,11.3,989.6,788.9,28,32.77


In [11]:
X , y = concrete.drop(['Strength'], axis=1), concrete['Strength']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.3, random_state=26)

In [15]:
dtr = DecisionTreeRegressor(random_state=26)
lr = LinearRegression()
ridge = Ridge()
rf = RandomForestRegressor(random_state=26)
xgbm = XGBRegressor(random_state=26)

stack = StackingRegressor(estimators= [('Tree', dtr),('RIDGE', ridge), ('LR', lr),('RF',rf)], final_estimator= xgbm)  

stack.fit(X_train, y_train)
y_pred = stack.predict(X_test)
r2_score(y_test, y_pred)

0.8617677109372424

In [16]:
dtr = DecisionTreeRegressor(random_state=26)
lr = LinearRegression()
ridge = Ridge()
rf = RandomForestRegressor(random_state=26)
xgbm = XGBRegressor(random_state=26)

stack = StackingRegressor(estimators= [('Tree', dtr),('RIDGE', ridge), ('LR', lr),('RF',rf)], final_estimator= xgbm, passthrough=True)  

stack.fit(X_train, y_train)
y_pred = stack.predict(X_test)
r2_score(y_test, y_pred)

0.9014182478659871

# Road Accident

In [30]:
os.chdir("C:/Users/PGCP-AI/Downloads/PredictingRoadAccident")

In [31]:
train = pd.read_csv('train.csv',index_col=0)
test = pd.read_csv('test.csv',index_col=0)
train

,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
id,,,,,,,,,,,,,
0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1,0.13
1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0,0.35
2,rural,4,0.63,70,dim,clear,False,True,morning,True,False,2,0.30
3,highway,4,0.07,35,dim,rainy,True,True,morning,False,False,1,0.21
4,rural,1,0.58,60,daylight,foggy,False,False,evening,True,False,1,0.56
...,...,...,...,...,...,...,...,...,...,...,...,...,...
517749,highway,4,0.10,70,daylight,foggy,True,True,afternoon,False,False,2,0.32
517750,rural,4,0.47,35,daylight,rainy,True,True,morning,False,False,1,0.26
517751,urban,4,0.62,25,daylight,foggy,False,False,afternoon,False,True,0,0.19


In [32]:
X , y = train.drop('accident_risk', axis=1), train['accident_risk']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.3, random_state=26)

In [33]:
ohe=OneHotEncoder(sparse_output=False,drop="first").set_output(transform="pandas")

trans = ColumnTransformer(transformers=[("OHE", ohe, make_column_selector(dtype_include=object))],     remainder="passthrough",verbose_feature_names_out=False).set_output(transform="pandas")

X_trn_ohe = trans.fit_transform(X_train)
X_tst_ohe = trans.transform(X_test)

In [35]:
dtr = DecisionTreeRegressor(random_state=26)
lr = LinearRegression()
ridge = Ridge()
rf = RandomForestRegressor(random_state=26)
xgbm = XGBRegressor(random_state=26)

stack = StackingRegressor(estimators= [('Tree', dtr),('RIDGE', ridge), ('LR', lr),('RF',rf)], final_estimator= xgbm, passthrough=True)  

stack.fit(X_trn_ohe, y_train)
y_pred = stack.predict(X_tst_ohe)

In [29]:
y_pred[y_pred<0]=0
submit=pd.read_csv('sample_submission.csv')
submit['accident_risk']=y_pred
submit.to_csv("sbt_stack_1.csv", index=False)

ValueError: Length of values (155327) does not match length of index (172585)